# Fused cross-entropy: loss and gradient in one kernel

For a language model with a 100k-token vocabulary, the logits of one batch can take gigabytes, and the standard pipeline (softmax, then log, then loss, then a separate backward pass) reads and writes them several times. A **fused** kernel reads each row of logits once and produces both the loss and the gradient.

For row $r$ with logits $z_{r,:}$ and target class $t_r$:

$$\ell_r = \log \sum_j e^{z_{r,j}} - z_{r,t_r}, \qquad \frac{\partial L}{\partial z_{r,j}} = \frac{\mathrm{softmax}(z_r)_j - [j = t_r]}{N_{\text{valid}}}$$

where $L = \frac{1}{N_{\text{valid}}} \sum_{r \text{ valid}} \ell_r$ is the mean over rows whose target is not `ignore_index = -100` (padding tokens). Ignored rows contribute 0 to the loss and have a zero gradient row.

## Task
Write `cross_entropy_fwd_bwd(logits, targets)` returning `(loss, grad)`.

- Kernel: **one block of 64 threads per row**. The block computes the row max and $\sum_j e^{z_j - m}$ with shared-memory tree reductions (the stable log-sum-exp $m + \log \sum_j e^{z_j - m}$), then thread 0 writes $\ell_r$ to a per-row `float32` array and all threads write the gradient row.
- The host counts $N_{\text{valid}}$ from `targets` before the launch and passes it in (the gradient needs it). It returns `loss = sum of per-row losses / N_valid` as a Python float computed in `float64` on the host, or `0.0` when no row is valid (all gradients 0 then).
- Ignored rows return from the kernel only after writing zeros; since the whole block takes the same branch, no barrier is skipped by part of a block.

## Inputs and output
- `logits`: `float32` `(N, C)`, $N \ge 1$, $C \ge 1$, finite values up to $\pm 10^3$. `targets`: `int64` `(N,)`, each in `[0, C)` or equal to `-100`.
- `grad`: `float32` `(N, C)`. Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

TPB = 64
IGNORE = -100

def cross_entropy_fwd_bwd(logits, targets):
    """logits: float32 (N, C), targets: int64 (N,). Returns (mean loss: float, grad: float32 (N, C))."""
    N, C = logits.shape
    n_valid = N                                   # TODO: rows with target -100 do not count
    d_z = cuda.to_device(np.ascontiguousarray(logits))
    d_t = cuda.to_device(targets)
    d_loss = cuda.device_array(N, dtype=np.float32)
    d_grad = cuda.device_array((N, C), dtype=np.float32)
    ce_kernel[N, TPB](d_z, d_t, n_valid, d_loss, d_grad)   # one block per row
    return float(d_loss.copy_to_host().astype(np.float64).sum() / n_valid), d_grad.copy_to_host()

@cuda.jit
def ce_kernel(z, targets, n_valid, loss, grad):
    r = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    buf = cuda.shared.array(TPB, float32)
    # TODO 1: ignored rows write zeros and return
    # TODO 2: block-wide max and sum of exp(z - max)
    # TODO 3: thread 0 writes loss[r]; every thread writes its columns of grad[r, :]
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[2.0, 1.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [0, 2], "dtype": "int64"}], "e": {"__tup__": [0.7531092166900635, {"__nd__": [[-0.16737955808639526, 0.12236422300338745, 0.045015282928943634], [0.1666666716337204, 0.1666666716337204, -0.3333333134651184]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 2.0], [3.0, 1.0], [0.5, 0.5]], "dtype": "float32"}, {"__nd__": [1, -100, 0], "dtype": "int64"}], "e": {"__tup__": [0.5032044649124146, {"__nd__": [[0.13447071611881256, -0.13447070121765137], [0.0, 0.0], [-0.25, 0.25]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1000.0, -1000.0, 999.0]], "dtype": "float32"}, {"__nd__": [2], "dtype": "int64"}], "e": {"__tup__": [1.313232421875, {"__nd__": [[0.7310585975646973, 0.0, -0.7310585975646973]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.9712313413619995, -0.1972997635602951, 1.2450041770935059, 1.7488224506378174, -0.5160723924636841], [1.29518723487854, -0.566789984703064, 0.05390108376741409, 1.072989821434021, -0.3285062313079834], [2.3725697994232178, -2.5347697734832764, -0.2980402112007141, -0.8106024861335754, -0.234228253364563]], "dtype": "float32"}, {"__nd__": [-100, -100, -100], "dtype": "int64"}], "e": {"__tup__": [0.0, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.07358841598033905], [-1.606993317604065], [1.1318610906600952], [-1.2350854873657227]], "dtype": "float32"}, {"__nd__": [0, 0, 0, 0], "dtype": "int64"}], "e": {"__tup__": [0.0, {"__nd__": [[0.0], [0.0], [0.0], [0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[2.2945711612701416, 3.9170033931732178, -4.53083610534668, 4.502366065979004, 0.9997920989990234, -1.3645999431610107, 3.2737038135528564, 6.388007640838623, 5.5027313232421875, 1.3673925399780273, -1.3045910596847534, -2.307281017303467, 3.053981065750122, 0.3451527953147888, -2.194723606109619, -2.784878969192505, -3.9719998836517334, -3.073638677597046, 1.5219676494598389, -3.6490464210510254, -4.550732135772705, -2.905836820602417, -4.0989227294921875, -4.122992992401123, 3.5694081783294678, 7.450189590454102, -0.025895986706018448, 1.4113949537277222, -0.9098500609397888, 2.1124026775360107, -4.701766014099121, 0.547269880771637, 5.773602485656738, -4.078212738037109, -1.1389307975769043, -1.1028354167938232, -3.7864162921905518, 2.7512171268463135, -1.33542799949646, 5.55047082901001, -0.28072553873062134, -2.4487967491149902, -3.525496006011963, -4.782840251922607, 2.7575178146362305, -0.9054816365242004, 6.463829517364502, 0.12394049763679504, 1.9926890134811401, 1.973867654800415, -5.29169225692749, 2.828646183013916, 1.0092494487762451, -2.9473607540130615, -0.6132852435112, 1.7869048118591309, 1.6325081586837769, 2.2137868404388428, 1.8679088354110718, 4.875949382781982, -0.6408954858779907, -1.5227439403533936, -1.3145101070404053, 1.4492642879486084, -2.4955477714538574, -4.969154357910156, -4.87283182144165, 4.959662437438965, 0.5483318567276001, 5.0857062339782715, 1.851010799407959, 0.004791246261447668, 6.2328619956970215, 5.010180473327637, 8.354873657226562, -4.860702037811279, -2.6277637481689453, 1.0928149223327637, -8.147970199584961, 1.340641975402832, -4.054100036621094, -3.039842367172241, -5.132997035980225, 3.926980972290039, 1.4862152338027954, -5.981189727783203, -7.345077991485596, 3.7383370399475098, 3.6350905895233154, -1.3873449563980103, 6.954412937164307, 0.519930899143219, 0.034566231071949005, 2.8030993938446045, -2.1431784629821777, -4.022232532501221, 3.523483991622925, -1.158020257949829, -1.0957762002944946, 4.576325416564941, 0.6846573948860168, -5.000413417816162, 2.7006571292877197, -1.1171855926513672, -0.12571993470191956, -1.4568253755569458, 5.55042839050293, 0.06874294579029083, 7.386601448059082, 3.5750558376312256, -0.42878249287605286, 6.355025291442871, 6.312924861907959, 0.3413105905056, -2.380976438522339, 1.5643106698989868, 0.38458576798439026, -5.0818772315979, 10.340712547302246, -4.489677429199219, 4.566613674163818, 0.8970960378646851, 0.31492525339126587, -0.0989617183804512, 0.18278366327285767, -6.4720659255981445, -6.142165184020996, 5.141443252563477, -0.6369090676307678, 6.395515441894531, 0.9065049290657043, -0.42978009581565857, -0.7301364541053772, -0.4864770770072937, -4.865246295928955, -0.39669567346572876, 3.346231698989868, 2.8869004249572754, -2.2421152591705322, 3.859362840652466, 0.5655069351196289, 0.6574319005012512, -12.22458553314209, -2.3002817630767822, -1.368653416633606, -3.3660390377044678, 3.000572681427002, -8.106322288513184, -0.3038741946220398, -0.39150550961494446, 4.479979515075684, -3.497263193130493, -0.12609612941741943, -2.9102840423583984, -0.6315377354621887, 0.8302499651908875, -2.0156335830688477, -0.19994692504405975, 0.4459182620048523, 4.915838241577148, -5.2854905128479, 2.4659557342529297, -5.302155017852783, -3.2960968017578125, -4.282503604888916, 3.271770477294922, -3.5726945400238037, 8.090749740600586, 0.3243447542190552, 3.0951948165893555, -1.1216628551483154, 3.6630122661590576, -4.7390313148498535, 1.6888347864151, -0.10657943040132523, 8.744041442871094, -4.072404861450195, -13.248249053955078, 6.602114200592041, -0.6962952613830566, -3.1063952445983887, 2.736086845397949, 0.7757890820503235, -3.1677370071411133, -7.581049919128418, 2.0817086696624756, -2.3631410598754883, -6.971573352813721, -3.353142023086548, -0.7892565727233887, -3.7781403064727783, -2.437329053878784, 5.530406475067139, -0.7882827520370483, 4.816118240356445, 2.7331061363220215, -6.349207401275635, 0.9099154472351074, -0.8016767501831055, -8.816797256469727], [-2.9533498287200928, -5.095233917236328, 8.225050926208496, -4.469451427459717, 1.5641123056411743, 2.0874669551849365, -1.9643480777740479, -0.40491783618927, -4.342276096343994, 1.1586312055587769, 8.318687438964844, 2.2979302406311035, 3.9542596340179443, -5.202380180358887, 2.8603744506835938, -2.525106191635132, -1.7015609741210938, -0.5837690234184265, 7.179938793182373, -6.80301570892334, 8.018599510192871, -5.309852600097656, 0.22912472486495972, -8.87183952331543, -8.158464431762695, 6.754335403442383, -1.7791811227798462, -3.2752420902252197, 8.578656196594238, -3.85894513130188, 0.21358232200145721, 3.6140685081481934, 1.2819385528564453, 1.7142962217330933, 0.19930855929851532, 0.06594165414571762, -3.873316526412964, 5.740789413452148, -0.6972938776016235, 0.7017268538475037, 3.490718126296997, -0.5326268076896667, -6.68513822555542, -8.713866233825684, 1.9233675003051758, 2.920109510421753, -1.4275603294372559, -0.7434578537940979, -1.784908652305603, 1.439858078956604, 6.159019470214844, -0.3906955122947693, 5.305789470672607, -1.1346044540405273, 1.0425831079483032, -5.061537265777588, -1.2861287593841553, 2.4888787269592285, -1.4474527835845947, -0.48258674144744873, 1.6608836650848389, -5.7653985023498535, -7.516918182373047, 5.086386203765869, -4.711770534515381, 4.3215484619140625, -4.920897483825684, 1.6040939092636108, -3.458779811859131, 4.123698711395264, 2.7513279914855957, 0.13383148610591888, 0.23578859865665436, -3.110623598098755, -1.3087457418441772, -3.977085828781128, -0.5105168223381042, 0.6007364988327026, -3.2480335235595703, -0.5560024976730347, -0.5202578902244568, 0.3848673403263092, -2.161327600479126, 0.8260639309883118, -4.268430233001709, -3.060523271560669, -0.7515302300453186, 2.913080930709839, 4.028521537780762, 0.2113512009382248, -2.2964706420898438, 0.3693047761917114, 0.6775633096694946, 8.785685539245605, 2.598984956741333, 5.575638771057129, 0.6726366877555847, 0.1749156266450882, -0.10923121869564056, 3.853341817855835, 0.5470617413520813, -0.813028872013092, 2.8400983810424805, 1.7222055196762085, 1.8770437240600586, 9.547842979431152, -4.499265193939209, 1.1903939247131348, -2.591075897216797, 2.30989933013916, 0.9111658334732056, -3.880053758621216, -1.6788661479949951, -1.1671228408813477, -5.344015121459961, -1.5937048196792603, -5.3610734939575195, -7.157662391662598, 5.025300979614258, -10.037967681884766, 7.931307315826416, -0.5402007699012756, -3.7949564456939697, 3.317312479019165, 7.299313545227051, -0.7293624877929688, 1.4262442588806152, -1.3879033327102661, 3.881220817565918, -1.1707104444503784, -2.427483320236206, -1.934906244277954, 0.5696195960044861, -5.618119239807129, -1.272584319114685, 4.697533130645752, 10.169910430908203, -5.029730796813965, -0.2738984525203705, -1.4337575435638428, -7.221407890319824, -0.5881661176681519, -3.1067423820495605, 0.3117009103298187, -2.807269334793091, 5.710421562194824, 0.5562651753425598, -4.316159725189209, -1.4093053340911865, 1.046350359916687, 2.0755510330200195, 8.088909149169922, -1.8111882209777832, -6.100709438323975, 4.358424186706543, 3.8771169185638428, 4.654605865478516, -4.486657619476318, 0.6173708438873291, 1.844639539718628, -0.6318897604942322, -1.951835036277771, 5.921652793884277, 5.203487873077393, -4.090494632720947, 2.241001844406128, 4.590600490570068, -1.8300840854644775, -4.209542751312256, -1.2312949895858765, 2.7265539169311523, 1.7244294881820679, -1.2141244411468506, 0.7380136847496033, -5.340270519256592, 0.6873536109924316, 3.2543816566467285, -1.7753510475158691, 1.2323546409606934, -1.596431016921997, -3.322911262512207, 1.8291457891464233, -3.602041721343994, -0.6981849670410156, 3.810112714767456, -3.5174551010131836, 4.510430812835693, 7.4973931312561035, 2.7705302238464355, 0.5798695683479309, 2.5680391788482666, -2.1216392517089844, -0.6370762586593628, 4.639614105224609, 2.3287320137023926, -3.774139881134033, -2.8385374546051025, -1.2927260398864746, 2.3026955127716064, 1.030537724494934], [3.430009603500366, 1.6919134855270386, 4.9107818603515625, -10.105401039123535, 1.2412937879562378, -7.0341925621032715, 6.785062789916992, 1.8199865818023682, 0.2055763155221939, -4.062110424041748, -3.176546335220337, -3.643115282058716, 4.373717308044434, -3.337752342224121, 5.620603084564209, -4.396419525146484, 2.512633800506592, 4.500015735626221, -0.6496130228042603, 2.3523144721984863, -6.48495626449585, 7.079270839691162, -4.761953353881836, 1.1298972368240356, 2.2293620109558105, 1.6461517810821533, 1.2951639890670776, -0.0988425761461258, 0.7464948296546936, -5.260218143463135, -3.9545578956604004, -2.4838507175445557, -5.389949798583984, 2.915837287902832, -2.816984176635742, 4.877673625946045, 0.29736024141311646, 4.224013805389404, -4.421870708465576, -4.167501926422119, 1.5250962972640991, -4.987838268280029, 2.677704334259033, 2.8529608249664307, -0.4141405522823334, 1.2437541484832764, -2.1886534690856934, 1.7676506042480469, -0.20909468829631805, 1.2316572666168213, -3.597703456878662, 0.8482260704040527, 3.304457187652588, -0.7820261716842651, -2.2251131534576416, 3.3335392475128174, -1.7067227363586426, -6.540043830871582, -2.808763265609741, 5.4823408126831055, -1.188520908355713, 10.762621879577637, 2.4109902381896973, 0.48101550340652466, -0.5236643552780151, -3.9224162101745605, 8.180479049682617, 1.3219807147979736, 5.374073028564453, -7.36670446395874, -2.124480724334717, 6.341254234313965, -1.7680599689483643, 0.4922046363353729, 1.7146570682525635, -2.778426170349121, -0.059029512107372284, -3.322939872741699, -2.0350992679595947, 0.5045722723007202, 3.773144483566284, -2.8636674880981445, 2.1269540786743164, -3.7775115966796875, -0.8808542490005493, -8.22435474395752, -2.1439316272735596, -1.3107504844665527, -1.2938330173492432, -3.67732834815979, 9.405146598815918, 0.16976642608642578, 2.0486299991607666, 2.938754081726074, 1.0518345832824707, 3.3474044799804688, -2.992259979248047, -3.0406582355499268, 5.821232318878174, -0.32161661982536316, -1.053284764289856, 5.965249538421631, -0.8287550806999207, -6.248915195465088, 2.7244982719421387, -1.3707213401794434, -6.056829452514648, 6.452896595001221, 5.447258472442627, 3.4446678161621094, 2.952298402786255, 7.77488899230957, 3.448920488357544, -1.0936964750289917, -0.9014836549758911, -6.22489070892334, 2.2076029777526855, 0.2739052176475525, 1.5590094327926636, -5.7716546058654785, 1.5349068641662598, -1.8641291856765747, -0.3367808163166046, -1.5561131238937378, 0.1624905914068222, -5.249582767486572, 4.2697434425354, -0.04863680899143219, -2.186532497406006, 4.630285739898682, -0.0766276940703392, 0.42679643630981445, 3.9976789951324463, 6.337220191955566, -1.4455454349517822, 0.012685923837125301, 0.32842764258384705, -4.228532791137695, -1.9658052921295166, -2.941568613052368, -2.78186297416687, 4.169009208679199, 7.967299938201904, 1.605488657951355, -2.8684229850769043, 1.2308751344680786, 3.120849370956421, 1.8460582494735718, -1.445242166519165, 4.785035610198975, 3.6247925758361816, -2.512005567550659, 1.6977554559707642, -5.989511489868164, -0.13672906160354614, 1.5857077836990356, 2.0934576988220215, 6.943827152252197, 0.022120727226138115, -1.3404622077941895, 4.377590656280518, -6.274359703063965, 1.9490476846694946, -0.721153199672699, -0.7394782900810242, -1.3228205442428589, -1.5097787380218506, -2.4995646476745605, 1.6304339170455933, 6.2018890380859375, -1.5926265716552734, 3.0848586559295654, 8.751572608947754, -2.252239942550659, -2.1095619201660156, -3.8788397312164307, 2.0567498207092285, 0.5843105912208557, 4.224518299102783, -1.257570505142212, 1.8602830171585083, -2.438842296600342, -5.006162166595459, -3.390575408935547, -1.7868050336837769, -7.203579425811768, -2.2027320861816406, -7.196639537811279, -3.992497205734253, 2.92966628074646, 0.5881412029266357, 2.7873549461364746, -3.6051511764526367, 5.399655342102051, -3.8538339138031006, -2.2280664443969727, 1.4604287147521973, -0.9885929226875305, 1.984845757484436, 2.488253355026245], [-5.803501605987549, 0.6314193606376648, 1.3033791780471802, -0.1869017630815506, 0.6173754334449768, -3.677729368209839, -2.244110107421875, -4.455218315124512, -0.24505653977394104, -4.2337117195129395, -4.821923732757568, -3.3919403553009033, -0.09414356201887131, 2.775618076324463, -4.84560489654541, 2.8608057498931885, 2.3498375415802, 0.012043578550219536, -6.982602596282959, -4.800310134887695, -0.10077192634344101, -6.549007415771484, -1.3284785747528076, 3.6200342178344727, -6.128737926483154, 1.9531843662261963, 0.17602424323558807, -3.658848285675049, 2.8558504581451416, 6.30649471282959, 0.46592339873313904, -1.5610257387161255, -0.8746678829193115, -0.6491533517837524, 6.118779182434082, -2.529454469680786, -5.033952236175537, 2.0066959857940674, -2.692418336868286, -0.13847577571868896, -3.4589571952819824, -5.606658935546875, -0.37858331203460693, -6.823094367980957, -0.11203153431415558, 4.15520715713501, -2.0885989665985107, -3.2344884872436523, 2.357539653778076, 3.6492128372192383, -4.322179317474365, 2.900442123413086, -2.977203369140625, -7.240307331085205, 2.581845998764038, -8.471891403198242, -0.6352317333221436, -5.453032493591309, -0.8544778823852539, -1.678617238998413, -0.36206546425819397, 3.713181972503662, 0.6066636443138123, 6.136125564575195, -6.588568687438965, 4.461803913116455, -1.2224206924438477, 6.37058162689209, 0.6036842465400696, -4.541914939880371, -7.452916145324707, -8.845118522644043, -3.7118589878082275, 3.85231876373291, 3.83426833152771, 0.12771281599998474, -5.819015979766846, -1.3835396766662598, -0.2716219127178192, -3.4078876972198486, -2.7404394149780273, -3.7530317306518555, 2.9767017364501953, -6.105295181274414, -1.3658969402313232, 8.72361946105957, -4.69997501373291, 0.21015752851963043, -2.780491828918457, 2.014993906021118, -5.439559459686279, -7.713311672210693, 4.061446666717529, -3.765650510787964, 3.608271598815918, 0.5545064806938171, 5.787992477416992, 0.24922862648963928, 5.668694019317627, 2.9581429958343506, -2.3572051525115967, 2.6812400817871094, 1.018692135810852, 4.735973834991455, 3.388047456741333, 3.4898812770843506, -0.9606863856315613, -2.7438535690307617, 7.479299545288086, 2.112752676010132, 2.6061806678771973, 5.734950542449951, -0.7276206612586975, 4.375509738922119, 2.222923517227173, -2.9207723140716553, 0.8503057360649109, -3.2565784454345703, 4.59879732131958, -5.713565349578857, -1.7886475324630737, -1.1294384002685547, -3.225135326385498, -0.04109586775302887, 3.623967170715332, 0.35216009616851807, 2.4767558574676514, -0.7347056269645691, 0.9983938932418823, 2.0700392723083496, 5.310759544372559, -3.2163448333740234, 5.841144561767578, -3.8400497436523438, -0.7570784687995911, 0.48264235258102417, -1.4974175691604614, -2.6431996822357178, 1.5587432384490967, 0.3600037693977356, -1.1476876735687256, -9.807369232177734, 3.302854537963867, -1.012810468673706, -7.666433334350586, -4.537532806396484, 5.17567253112793, 11.216329574584961, -1.0945433378219604, 1.881850242614746, 2.9809701442718506, 4.998380661010742, 6.784054756164551, 0.6650958061218262, -3.3983936309814453, 1.5080883502960205, 1.9567509889602661, 2.852773427963257, 6.19685173034668, 0.7123969793319702, -1.4397000074386597, 4.905818462371826, 1.0065560340881348, -2.9124553203582764, -2.125250816345215, 2.7756552696228027, -8.72287654876709, 4.630631923675537, -1.7337958812713623, -4.757977485656738, 1.1220706701278687, -1.9650269746780396, -2.9166040420532227, -3.167872905731201, 3.094799280166626, -4.9362406730651855, -3.1183090209960938, -9.525887489318848, -0.6102598905563354, 2.2403597831726074, 3.002229690551758, -0.08704414963722229, 9.403263092041016, 7.160618782043457, 0.4552956223487854, 4.801066875457764, 3.100092649459839, 3.806180477142334, -5.231197357177734, 0.7526866793632507, 0.9091851115226746, 2.2380330562591553, -4.731133937835693, 1.4876362085342407, -2.8933935165405273, -1.6159696578979492, 4.878237724304199, 0.6684792041778564, -2.058192014694214, 2.1142325401306152], [-1.796212911605835, -1.651060938835144, -3.077620506286621, 1.3585283756256104, -4.251121520996094, -1.843765139579773, 13.880148887634277, 3.6752519607543945, -3.286050319671631, 1.0145996809005737, 3.9497790336608887, -0.8707934021949768, -1.9112399816513062, -4.223038673400879, -5.257723331451416, 1.498997449874878, -3.3286936283111572, 0.22565297782421112, 1.3871736526489258, 3.802870988845825, -0.06481146812438965, 0.13325516879558563, -3.824199676513672, -0.929807722568512, 0.6427703499794006, -2.7463877201080322, 2.067049264907837, -2.0704991817474365, -2.001894235610962, -3.524016857147217, -6.246730804443359, -0.06793934851884842, 6.509528636932373, -1.400116205215454, -1.8147892951965332, -5.34414005279541, 2.083879232406616, -1.3215899467468262, -3.6922709941864014, -6.418765068054199, 0.47550562024116516, -1.7108322381973267, 1.8925012350082397, 1.9395710229873657, -2.5158181190490723, 1.5340925455093384, 2.2601656913757324, 5.428788661956787, -1.73259437084198, -3.483970880508423, 3.1077775955200195, -1.1339681148529053, 9.14234447479248, -4.719388008117676, -8.893843650817871, 1.4809832572937012, -3.0962111949920654, 1.7206510305404663, 5.739459037780762, 2.3638153076171875, -0.33290404081344604, -7.118764400482178, -3.3134713172912598, -3.323047399520874, -2.4209773540496826, 1.504589319229126, -3.3528904914855957, -2.885624885559082, -0.6392546892166138, 2.7752084732055664, -2.179936408996582, -0.3885357975959778, -0.07404190301895142, -10.13693618774414, 0.211165189743042, 2.52341365814209, -0.6902596354484558, -1.4300135374069214, -3.0553932189941406, -7.629828929901123, -2.6834990978240967, -0.3471194803714752, 1.5156694650650024, 3.4915995597839355, -10.89144229888916, -3.904405117034912, -0.7445825338363647, -0.4386472702026367, -1.4919921159744263, -2.3679614067077637, 1.426493763923645, 3.0765013694763184, 1.5246435403823853, -6.773230075836182, -4.046263217926025, 2.613856554031372, 0.11634407192468643, 3.3407349586486816, 6.161352157592773, 3.0197834968566895, -0.21877014636993408, -3.559419870376587, 2.5096914768218994, -2.7803735733032227, -6.030158996582031, -3.308199882507324, 5.822827339172363, 2.859013795852661, 0.24629147350788116, 9.358206748962402, 8.624753952026367, 2.3865973949432373, 8.361157417297363, -2.4238173961639404, -1.7860300540924072, -2.5876150131225586, -1.4396569728851318, -1.880471110343933, -4.438655376434326, 12.304018020629883, -0.4378426671028137, 9.791634559631348, 0.7121407985687256, 4.584149360656738, 6.177774906158447, -2.1458938121795654, 0.011434764601290226, 2.455022096633911, 2.6320741176605225, -1.316980004310608, 2.4842112064361572, 3.3192315101623535, 4.303643703460693, -7.79431676864624, 1.9938267469406128, -2.568333148956299, -2.8202390670776367, 6.219060897827148, 2.3763363361358643, -0.7404406070709229, 6.272657871246338, 0.9526718258857727, 0.6281967163085938, 0.8735299706459045, 2.181936264038086, 0.3719536066055298, -2.7905197143554688, -0.9530189633369446, 2.730689287185669, -7.444908618927002, -4.685389995574951, -0.5221055746078491, -1.5182855129241943, -3.05184268951416, 1.7273333072662354, -2.4057528972625732, 1.4585111141204834, 0.8878635168075562, -0.18729321658611298, 0.08882374316453934, -1.6537810564041138, -3.6305224895477295, -0.04504575952887535, 7.0844316482543945, 7.030800819396973, 4.2187418937683105, -0.042287785559892654, -3.525636672973633, 2.7288787364959717, 3.2966063022613525, -1.2701176404953003, 14.072598457336426, -5.928895473480225, -1.4774874448776245, 2.4977176189422607, 6.101206302642822, -1.2919410467147827, -0.08849354833364487, -0.2527584433555603, 0.30927029252052307, 6.821023941040039, 1.9604976177215576, -2.8686420917510986, 0.8936511278152466, 1.0168423652648926, -7.624460697174072, -8.89270305633545, 2.012448787689209, 6.8037190437316895, 0.12667103111743927, 0.8076981902122498, -3.380370855331421, -4.507875919342041, -3.9473912715911865, 0.30971431732177734, -2.5566649436950684, 2.274329423904419, -4.304414749145508, -6.493412971496582, -5.858051300048828], [-5.1640625, 0.2682698965072632, -2.1343295574188232, -2.3078463077545166, -12.95514965057373, -1.6484674215316772, 1.6575920581817627, -0.23992058634757996, -8.580588340759277, 4.5072808265686035, 3.001600980758667, -9.339414596557617, 4.069722652435303, -3.062323570251465, -0.21056312322616577, -5.922910690307617, 1.9332458972930908, 2.4783825874328613, 0.7484015822410583, 0.6276998519897461, -5.450634479522705, 2.490018129348755, -7.451974391937256, -0.5602057576179504, 2.720207691192627, 3.1371328830718994, -1.3054263591766357, 5.783665180206299, 2.3271613121032715, 0.24848337471485138, -5.68237829208374, 1.2461527585983276, 0.5012422800064087, -0.5013036131858826, 4.851413249969482, -8.980658531188965, -2.2006592750549316, -2.8188693523406982, -5.1685051918029785, -3.999006986618042, -0.37173211574554443, 3.6570258140563965, 1.3217893838882446, -1.5652364492416382, -1.42936372756958, 3.6350347995758057, -7.899247169494629, -3.0206830501556396, 3.5703563690185547, -0.7109028100967407, -2.9466869831085205, 1.8269612789154053, 1.2680233716964722, -0.6019799113273621, -2.329639434814453, -5.3456220626831055, -4.562084674835205, 0.8281422257423401, 4.332198619842529, -0.18521496653556824, -2.472046136856079, 4.8745927810668945, 7.225207328796387, 0.8212977051734924, 2.0224058628082275, 1.532240629196167, 1.8719924688339233, -7.633963108062744, -0.2457432895898819, -3.5223820209503174, 4.162049770355225, 2.5530858039855957, 3.5425941944122314, -11.824941635131836, -0.020459193736314774, -5.617107391357422, 2.983144521713257, -6.9608917236328125, -2.6889262199401855, -1.546302318572998, -4.342515468597412, 0.7908583283424377, -3.7157936096191406, -1.9089573621749878, -1.499800443649292, -2.577404260635376, 7.656625270843506, 4.027245044708252, -2.026609420776367, -4.4981184005737305, -2.570310115814209, -2.421497106552124, -0.058115001767873764, 2.182487964630127, -5.913049697875977, -1.4495463371276855, -3.1797986030578613, 2.424760103225708, 0.6051968336105347, 8.766739845275879, -2.9810597896575928, 0.40631556510925293, -0.6231778860092163, 7.546022891998291, 0.834342360496521, 1.431149959564209, -2.333946704864502, 2.5781805515289307, -5.220250606536865, -0.7971721887588501, -0.14095181226730347, 5.4337944984436035, -2.8849947452545166, -0.5683283805847168, 0.13828317821025848, 0.477865070104599, 10.62900447845459, 4.902209281921387, -3.532496452331543, -3.5959722995758057, 2.766187906265259, -4.174670696258545, -2.8804399967193604, 6.990484714508057, 1.9517470598220825, 4.7220025062561035, 10.525005340576172, -1.4454046487808228, -1.7431252002716064, -2.984255790710449, 3.207794427871704, -3.571171998977661, -4.494130611419678, -0.5682662725448608, -8.379679679870605, 10.51935863494873, 4.575636386871338, -3.7779603004455566, 0.5658576488494873, -10.502850532531738, -0.9956448674201965, 2.405642032623291, 1.7659896612167358, -0.41399848461151123, 1.2356599569320679, -4.866062164306641, -10.17566967010498, 1.0184143781661987, -8.273090362548828, 2.8015575408935547, -2.9370882511138916, 2.157400131225586, -1.882041096687317, -0.971145510673523, -2.1950933933258057, -4.0145649909973145, -0.685728132724762, -2.0205116271972656, -3.5537705421447754, -0.8359388113021851, -2.0318052768707275, 0.19241556525230408, -0.39774271845817566, 1.845414161682129, 2.112677574157715, 13.411355018615723, -0.6760730743408203, -2.6434166431427, 0.5877845883369446, -2.228457450866699, -0.2637518346309662, 4.200805187225342, 1.3130165338516235, 8.471963882446289, -0.32995733618736267, -1.1793179512023926, 4.852509498596191, -2.3922805786132812, -0.2834608256816864, 2.103618860244751, 6.065445423126221, -0.9638702273368835, -5.719330310821533, 7.530913352966309, -4.0248870849609375, -5.383668422698975, 1.7186918258666992, -1.3467124700546265, -3.5051674842834473, -2.0812690258026123, 0.7094022631645203, 5.538409233093262, 3.4188380241394043, 5.350767135620117, 4.723295211791992, -1.807303547859192, -2.73618745803833, -0.6899653077125549, -0.9813965559005737, 3.109473705291748]], "dtype": "float32"}, {"__nd__": [103, 7, 199, 70, 157, 191], "dtype": "int64"}], "e": {"__tup__": [12.256125926971436, {"__nd__": [[2.9056482162559405e-05, 0.00014718263992108405, 3.155051331305003e-08, 0.0002642872859723866, 7.960268703754991e-06, 7.483162676180655e-07, 7.73526044213213e-05, 0.0017418069764971733, 0.0007186697912402451, 1.1496736078697722e-05, 7.945967581690638e-07, 2.915307106832188e-07, 6.209412822499871e-05, 4.136390089115594e-06, 3.262623522459762e-07, 1.808281666626499e-07, 5.517035717161889e-08, 1.3547499122523732e-07, 1.3418556591204833e-05, 7.620133857244582e-08, 3.0928969607657564e-08, 1.6022657689518383e-07, 4.8594152701753046e-08, 4.743846560018028e-08, 0.00010396756988484412, 0.00503848260268569, 2.8541512619995046e-06, 1.2013916602882091e-05, 1.1791814813477686e-06, 2.4217451937147416e-05, 2.6593317770107205e-08, 5.062905529484851e-06, 0.0009422523435205221, 4.961102817446772e-08, 9.377614560435177e-07, 9.7222789463558e-07, 6.64207746581269e-08, 4.587349758367054e-05, 7.704676932007715e-07, 0.0007538108620792627, 2.2121062102087308e-06, 2.5306079010078975e-07, 8.622235725397331e-08, 2.4522380570601854e-08, 4.616345904651098e-05, 1.1843443417092203e-06, 0.001879009767435491, 3.3155088203784544e-06, 2.1485102479346097e-05, 2.1084497348056175e-05, 1.4742496112774006e-08, 4.9566577217774466e-05, 8.035910468606744e-06, 1.537095783987752e-07, 1.5862704003666295e-06, 1.748905015119817e-05, 1.4986936548666563e-05, 2.6801495550898835e-05, 1.896469620987773e-05, 0.0003839913406409323, 1.5430714483954944e-06, 6.388577276084106e-07, 7.867540148254193e-07, 1.2477601558202878e-05, 2.415021924662142e-07, 2.035388035892538e-08, 2.2411933642274562e-08, 0.00041752029210329056, 5.068286554887891e-06, 0.0004736065457109362, 1.8646920580067672e-05, 2.943095296359388e-06, 0.0014914924977347255, 0.0004391544498503208, 0.012450851500034332, 2.268546417383277e-08, 2.115924928602908e-07, 8.736285053601023e-06, 8.474331791852308e-10, 1.1193269529030658e-05, 5.082182852333972e-08, 1.40131788839426e-07, 1.7277919539537834e-08, 0.000148658495163545, 1.294728645007126e-05, 7.398189172391767e-09, 1.8914569910322143e-09, 0.00012310130114201456, 0.00011102567805210128, 7.31487887151161e-07, 0.0030689279083162546, 4.926365363644436e-06, 3.0320429686980788e-06, 4.8316363972844556e-05, 3.4352089528510987e-07, 5.246744549936011e-08, 9.930093074217439e-05, 9.200297199640772e-07, 9.79115270638431e-07, 0.00028457478038035333, 5.80853111387114e-06, 1.9727488975718188e-08, 4.361179162515327e-05, -0.166665717959404, 2.58299769484438e-06, 6.823895546403946e-07, 0.0007537788478657603, 3.137459543722798e-06, 0.004728068597614765, 0.00010455640585860237, 1.907680598378647e-06, 0.0016852951375767589, 0.0016158162616193295, 4.120527137274621e-06, 2.708186173094873e-07, 1.3998940630699508e-05, 4.302757133700652e-06, 1.8184111993946317e-08, 0.09070640057325363, 3.287618355329869e-08, 0.00028182443929836154, 7.183353318396257e-06, 4.013227680843556e-06, 2.6530472041486064e-06, 3.516459628372104e-06, 4.528360797451114e-09, 6.298180643682372e-09, 0.0005007534637115896, 1.549234866615734e-06, 0.0017549332696944475, 7.251260285556782e-06, 1.9057786175835645e-06, 1.411331822964712e-06, 1.800732889023493e-06, 2.2582586467478905e-08, 1.969883669517003e-06, 8.317128231283277e-05, 5.2539806347340345e-05, 3.1116113063944795e-07, 0.0001389387616654858, 5.156085080670891e-06, 5.652524123433977e-06, 1.4376510398816933e-11, 2.935782958957134e-07, 7.452894692505652e-07, 1.0112795933991947e-07, 5.886480357730761e-05, 8.83471407142622e-10, 2.1614873730868567e-06, 1.9801359485427383e-06, 0.0002584365429356694, 8.869140089018401e-08, 2.5820272639975883e-06, 1.5951562204463698e-07, 1.5575798215650138e-06, 6.71887300995877e-06, 3.902518415088707e-07, 2.3982115635590162e-06, 4.57491796623799e-06, 0.0003996179439127445, 1.4834193429180687e-08, 3.448851566645317e-05, 1.4589038421775058e-08, 1.0845432996120508e-07, 4.0444145810170085e-08, 7.72032217355445e-05, 8.224734671102851e-08, 0.009560740552842617, 4.051208634336945e-06, 6.470674998126924e-05, 9.540946166453068e-07, 0.00011416938650654629, 2.5620533250503286e-08, 1.5855332094361074e-05, 2.6329130378144328e-06, 0.0183743704110384, 4.9900002352387673e-08, 5.165134812912253e-12, 0.0021576713770627975, 1.4599100950363209e-06, 1.311092745481801e-07, 4.518465357250534e-05, 6.362745352817001e-06, 1.2330845322594541e-07, 1.4938789094642857e-09, 2.3485414203605615e-05, 2.7569214466893754e-07, 2.7479416520037603e-09, 1.0244070836051833e-07, 1.3303133528097533e-06, 6.697275267697478e-08, 2.559793017553602e-07, 0.0007388368248939514, 1.3316093827597797e-06, 0.00036169050144962966, 4.505017204792239e-05, 5.12032549693231e-09, 7.276032192748971e-06, 1.3138921985955676e-06, 4.3414774553163227e-10], [1.1107130859500103e-07, 1.3043482738339662e-08, 0.007949132472276688, 2.4387498243072514e-08, 1.0174452654609922e-05, 1.7171214494737796e-05, 2.986208187394368e-07, -0.16666525602340698, 2.7694859738858213e-08, 6.78285823596525e-06, 0.008729423396289349, 2.119356213370338e-05, 0.00011105517478426918, 1.171818997391938e-08, 3.719378219102509e-05, 1.7044558831003087e-07, 3.883711769958609e-07, 1.1876759344886523e-06, 0.00279533164575696, 2.3643564883712997e-09, 0.006466347724199295, 1.0524122018296111e-08, 2.6775235255627194e-06, 2.986995650822166e-10, 6.09606309875943e-10, 0.0018264009850099683, 3.593658846057224e-07, 8.050189137520647e-08, 0.011321098543703556, 4.490627603104258e-08, 2.6362283733760705e-06, 7.903065124992281e-05, 7.672985702811275e-06, 1.1823198292404413e-05, 2.5988672405219404e-06, 2.2743836325389566e-06, 4.426550503922044e-08, 0.000662855280097574, 1.0602163911244133e-06, 4.295182861824287e-06, 6.985943036852404e-05, 1.2499957620093483e-06, 2.660153430866785e-09, 3.4981714747139847e-10, 1.4572478903573938e-05, 3.9483267755713314e-05, 5.10791608121508e-07, 1.0123854963239864e-06, 3.573137519197189e-07, 8.985621207102668e-06, 0.0010070562129840255, 1.4406172113012872e-06, 0.000429042789619416, 6.846570386187523e-07, 6.0396778280846775e-06, 1.3490499384261057e-08, 5.883920266569476e-07, 2.5652627300587483e-05, 5.007314598515222e-07, 1.3141371937308577e-06, 1.1208261639694683e-05, 6.673366303999728e-09, 1.1578956682356534e-09, 0.0003445204347372055, 1.9139447360316808e-08, 0.00016034305735956877, 1.552769113288832e-08, 1.0589476914901752e-05, 6.700332022546718e-08, 0.00013156038767192513, 3.3351250749547035e-05, 2.434151610941626e-06, 2.6954239729093388e-06, 9.490716479376715e-08, 5.752341394327232e-07, 3.9902399606717154e-08, 1.2779408962160232e-06, 3.882595592585858e-06, 8.272228058103792e-08, 1.2211152125018998e-06, 1.265553123630525e-06, 3.1287543151847785e-06, 2.4522967123630224e-07, 4.863854883296881e-06, 2.9817389446407105e-08, 9.978316484193783e-08, 1.0042456324299565e-06, 3.920672679669224e-05, 0.00011961626296397299, 2.63035440184467e-06, 2.1423034013423603e-07, 3.0804383186477935e-06, 4.1926409721781965e-06, 0.013925163075327873, 2.8638498406508006e-05, 0.0005619461298920214, 4.172036369709531e-06, 2.536239662731532e-06, 1.908917283799383e-06, 0.00010039468907052651, 3.6796923268411774e-06, 9.443465387448668e-07, 3.64472325600218e-05, 1.1917076335521415e-05, 1.3912823305872735e-05, 0.029840152710676193, 2.3671145044090736e-08, 7.001754056545906e-06, 1.595642373786177e-07, 2.144875361409504e-05, 5.295902610669145e-06, 4.396830277642039e-08, 3.9728587353238254e-07, 6.627512334489438e-07, 1.017066253439225e-08, 4.3260166648906306e-07, 9.99863924988631e-09, 1.6584129625130117e-09, 0.0003241052327211946, 9.306610343484678e-11, 0.005925821140408516, 1.2405639608914498e-06, 4.787370144754277e-08, 5.8737568906508386e-05, 0.003149757394567132, 1.0267564221067005e-06, 8.864122719387524e-06, 5.314553845892078e-07, 0.00010323294554837048, 6.603776796509919e-07, 1.879242432778483e-07, 3.0754347335459897e-07, 3.763639369935845e-06, 7.73227615269434e-09, 5.964161005067581e-07, 0.00023352760763373226, 0.05558549985289574, 1.3926475084247159e-08, 1.6190978158192593e-06, 5.07636059410288e-07, 1.5559925570229893e-09, 1.1824647572211688e-06, 9.527615674187473e-08, 2.908007218138664e-06, 1.2854169995080156e-07, 0.0006430282955989242, 3.71371265828202e-06, 2.84276513440318e-08, 5.202018655836582e-07, 6.062472493795212e-06, 1.6967813280643895e-05, 0.006937358528375626, 3.480460293303622e-07, 4.772224837523709e-09, 0.00016636621148791164, 0.00010281018330715597, 0.00022371503291651607, 2.3971470142214457e-08, 3.947718596464256e-06, 1.3469219084072392e-05, 1.131877638727019e-06, 3.0238101089707925e-07, 0.0007942672818899155, 0.00038732148823328316, 3.562431416526124e-08, 2.0020752344862558e-05, 0.00020984468574170023, 3.4153106298617786e-07, 3.162599071515615e-08, 6.215571488610294e-07, 3.253515387768857e-05, 1.1943609024456237e-05, 6.32321416560444e-07, 4.453901965462137e-06, 1.0208814238410469e-08, 4.233886556903599e-06, 5.515508382814005e-05, 3.607452185860893e-07, 7.301810455828672e-06, 4.314241550673614e-07, 7.67544392488162e-08, 1.3262130778457504e-05, 5.806024816479294e-08, 1.059272335623973e-06, 9.614718146622181e-05, 6.318502698832162e-08, 0.00019367820641491562, 0.0038397416938096285, 3.3997854188783094e-05, 3.802415221798583e-06, 2.7765830964199267e-05, 2.551580848830781e-07, 1.1260217434028164e-06, 0.00022038616589270532, 2.185652010666672e-05, 4.888072169251245e-08, 1.245846306119347e-07, 5.845229225087678e-07, 2.129479071300011e-05, 5.967361175862607e-06], [6.157235475257039e-05, 1.0827799087564927e-05, 0.00027069292264059186, 8.14763465029067e-11, 6.899832442286424e-06, 1.7572786559227893e-09, 0.0017638708231970668, 1.2307263205002528e-05, 2.4492433112754952e-06, 3.432412754023062e-08, 8.321373456965375e-08, 5.218752008318006e-08, 0.00015820955741219223, 7.082458353124821e-08, 0.0005504885921254754, 2.457031555991307e-08, 2.4602215489721857e-05, 0.00017950785695575178, 1.041424866343732e-06, 2.0957952074240893e-05, 3.043483465248187e-09, 0.00236722594127059, 1.704752605746762e-08, 6.172477242216701e-06, 1.8533219190430827e-05, 1.0343463145545684e-05, 7.281721536855912e-06, 1.806444515750627e-06, 4.2067840695381165e-06, 1.0357799951066227e-08, 3.822157168542617e-08, 1.6635233635042823e-07, 9.097574249494755e-09, 3.681997623061761e-05, 1.1922044507173268e-07, 0.0002618775179143995, 2.684685568965506e-06, 0.00013621262041851878, 2.3952873462462776e-08, 3.0890728197618955e-08, 9.164150469587184e-06, 1.3600693016257992e-08, 2.9017761335126124e-05, 3.457614002400078e-05, 1.3179302413846017e-06, 6.916829988767859e-06, 2.234763343267332e-07, 1.1679717317747418e-05, 1.6178663599930587e-06, 6.833658972027479e-06, 5.461210150770057e-08, 4.657267254515318e-06, 5.43073947483208e-05, 9.122665574068378e-07, 2.1547513995301415e-07, 5.5909946240717545e-05, 3.618519883730187e-07, 2.8803583962400126e-09, 1.2020456097161514e-07, 0.0004794042033609003, 6.075516125747527e-07, 0.09416700154542923, 2.2224468921194784e-05, 3.2259208637697157e-06, 1.181209313472209e-06, 3.94700734318576e-08, 0.007120137102901936, 7.479632586182561e-06, 0.00043021119199693203, 1.2601798493605543e-09, 2.3828756923194305e-07, 0.0011316786985844374, 3.4032422036034404e-07, 3.262219706812175e-06, 1.1076879673055373e-05, 1.2390712811338744e-07, 1.8798154997057281e-06, 7.188148032355457e-08, 2.605668498745217e-07, 3.302814775452134e-06, 8.677754522068426e-05, 1.1378275388551629e-07, 1.6729227354517207e-05, 4.5624592104331896e-08, 8.264206599051249e-07, 5.345145992485811e-10, 2.3369739210465923e-07, 5.376504077503341e-07, 5.468233439387404e-07, 5.043222728318142e-08, 0.024230072274804115, 2.3630889245396247e-06, 1.5468922356376424e-05, 3.767351518035866e-05, 5.708967819373356e-06, 5.669056190527044e-05, 1.0005284423186822e-07, 9.532580236282229e-08, 0.0006727915606461465, 1.4456904864346143e-06, 6.955298204047722e-07, 0.000777009641751647, 8.706178391548747e-07, 3.853735108805267e-09, 3.040785486518871e-05, 5.063548087491654e-07, 4.6698644950993184e-09, 0.001265344675630331, 0.0004628771566785872, 6.24815293122083e-05, 3.818723780568689e-05, 0.0047461651265621185, 6.274780025705695e-05, 6.679824764432851e-07, 8.095470889202261e-07, 3.947441928886519e-09, 1.813430935726501e-05, 2.6224474822811317e-06, 9.480269909545314e-06, 6.210885139523725e-09, 9.25450603972422e-06, 3.0915086313143547e-07, 1.423932872057776e-06, 4.2066852756761364e-07, 2.345958364458056e-06, 1.0468546918218635e-08, 0.00014258619921747595, 1.8994546735484619e-06, 2.2395082055481907e-07, 0.00020448383293114603, 1.8470236682333052e-06, 3.055673914786894e-06, 0.00010862286580959335, 0.0011271227849647403, 4.698499083133356e-07, 2.0195786873955512e-06, 2.7693995434674434e-06, 2.9061830275622924e-08, 2.792628492898075e-07, 1.0525543103767632e-07, 1.2348198197287275e-07, 0.00012892262020613998, 0.00575315160676837, 9.931306522048544e-06, 1.1324300430715084e-07, 6.8283166001492646e-06, 4.519800131674856e-05, 1.263235117221484e-05, 4.6999241476441966e-07, 0.00023870746372267604, 7.481330976588652e-05, 1.6173395067653473e-07, 1.08912427094765e-05, 4.9950528158149154e-09, 1.7392846984876087e-06, 9.73678379523335e-06, 1.6178131772903726e-05, 0.002067365450784564, 2.0387230961205205e-06, 5.219103513809387e-07, 0.00015882356092333794, 3.756917443809016e-09, 1.4002715033711866e-05, 9.695240805740468e-07, 9.519193895357603e-07, 5.311995892043342e-07, 4.406185496463877e-07, 1.6375875588892086e-07, 1.0182160622207448e-05, 0.0009844590676948428, 4.055855811202491e-07, 4.360021557658911e-05, 0.012604080140590668, 2.0970846037471347e-07, 2.4186905989154184e-07, 4.1228055636111094e-08, 1.559504926262889e-05, 3.5769614896707935e-06, 0.00013628134911414236, 5.670164000548539e-07, 1.2813330613425933e-05, 1.740106796432883e-07, 1.3353743888444569e-08, 6.718050116205632e-08, 3.3400411325601453e-07, 1.4834620198911352e-09, 2.203520352850319e-07, 1.4937944214921117e-09, 3.679864590822035e-08, 3.7332705687731504e-05, 3.590690994315082e-06, 3.238055796828121e-05, 5.420685056378716e-08, 0.000441358977695927, 4.22719743653488e-08, 2.1483967316271446e-07, 8.590289326093625e-06, 7.420120482493076e-07, 1.451306525268592e-05, -0.1666426658630371], [4.892705796066821e-09, 3.0492951736960094e-06, 5.970736765448237e-06, 1.3452626035359572e-06, 3.0067708394199144e-06, 4.0997740313741815e-08, 1.7193843859786284e-07, 1.8840866644609378e-08, 1.2692613609033288e-06, 2.3512573221751154e-08, 1.3056973280356488e-08, 5.4560295836836303e-08, 1.4760172462047194e-06, 2.602633321657777e-05, 1.275139549505866e-08, 2.8340640710666776e-05, 1.7001959349727258e-05, 1.6413764569733758e-06, 1.5047751933394693e-09, 1.3342259741477847e-08, 1.4662665535070118e-06, 2.3215593891734443e-09, 4.29562732051636e-07, 6.055339690647088e-05, 3.5342748727629214e-09, 1.1434961379563902e-05, 1.9338556285219966e-06, 4.177920587267181e-08, 2.8200549422763288e-05, 0.000888901122380048, 2.5841964088613167e-06, 3.4043387131532654e-07, 6.762605266885657e-07, 8.47332216835639e-07, 0.0007367660873569548, 1.2925566750254802e-07, 1.0562328789376352e-08, 1.206353226734791e-05, 1.0981844411617203e-07, 1.4120123523753136e-06, 5.1023629765722944e-08, 5.957122795052783e-09, 1.110608764065546e-06, 1.7650010342151745e-09, 1.4498489235847956e-06, 0.00010340967128286138, 2.008678876563863e-07, 6.386414241887906e-08, 1.713342862785794e-05, 6.234628381207585e-05, 2.1521810111835293e-08, 2.9486527637345716e-05, 8.260277439831043e-08, 1.1629265328494398e-09, 2.1441697754198685e-05, 3.393766101478235e-10, 8.592110702920763e-07, 6.946340835867204e-09, 6.900531843712088e-07, 3.0266573958215304e-07, 1.1291057262496906e-06, 6.646485417149961e-05, 2.9747352527920157e-06, 0.0007496578036807477, 2.2315107539583323e-09, 0.00014051231846679002, 4.776250079885358e-07, 0.0009477331186644733, 2.9658856419700896e-06, 1.7276232000540404e-08, -0.1666666716337204, 2.336640325673045e-10, 3.962212957731026e-08, 7.638681563548744e-05, 7.502038351958618e-05, 1.8426494534651283e-06, 4.817383825184152e-09, 4.065501855166076e-07, 1.2359867014311021e-06, 5.369710720515286e-08, 1.046694393380676e-07, 3.802389869633771e-08, 3.182311775162816e-05, 3.6181087015307867e-09, 4.137861822073319e-07, 0.009967755526304245, 1.475042044063457e-08, 2.0010033949802164e-06, 1.0056002253122642e-07, 1.2164051440777257e-05, 7.040552141290846e-09, 7.246506728009194e-10, 9.415458043804392e-05, 3.7547085440792216e-08, 5.984530071145855e-05, 2.8235574518475914e-06, 0.0005292621208354831, 2.0807319742743857e-06, 0.00046974277938716114, 3.1237977964337915e-05, 1.5355229265878734e-07, 2.3682381652179174e-05, 4.4914850150235e-06, 0.00018483528401702642, 4.801620525540784e-05, 5.316354145179503e-05, 6.2052123439571e-07, 1.0431268293586982e-07, 0.0028720772825181484, 1.341326606052462e-05, 2.196985951741226e-05, 0.0005019205855205655, 7.833859854144976e-07, 0.00012889536446891725, 1.4975486010371242e-05, 8.739814205682706e-08, 3.7954266645101598e-06, 6.24688567540943e-08, 0.00016114242316689342, 5.3531263866091194e-09, 2.7113014766655397e-07, 5.241658982413355e-07, 6.446424549721996e-08, 1.5564312434435124e-06, 6.079201557440683e-05, 2.306316901012906e-06, 1.930274629557971e-05, 7.778554049764352e-07, 4.401235401019221e-06, 1.2852406143792905e-05, 0.00032840605126693845, 6.503344707198266e-08, 0.0005581545410677791, 3.48550059925401e-08, 7.606462872900011e-07, 2.627764388307696e-06, 3.6279180903875385e-07, 1.1535873056800483e-07, 7.707814802415669e-06, 2.3244760996021796e-06, 5.146868033989449e-07, 8.926706596135148e-11, 4.409497705637477e-05, 5.890057082069688e-07, 7.594302409152931e-10, 1.7352103753864867e-08, 0.00028690864564850926, 0.12055011838674545, 5.427794462775637e-07, 1.06476745713735e-05, 3.195925091858953e-05, 0.00024029611086007208, 0.0014330295380204916, 3.1537354061583756e-06, 5.420932680522128e-08, 7.327097137022065e-06, 1.1475819519546349e-05, 2.8113923690398224e-05, 0.000796592328697443, 3.306492317278753e-06, 3.843472313747043e-07, 0.00021905213361606002, 4.437303687154781e-06, 8.812806129299133e-08, 1.9363903902558377e-07, 2.6027326384792104e-05, 2.6404678443725516e-10, 0.00016635486099403352, 2.864174746264325e-07, 1.3919198238454555e-08, 4.980655830877367e-06, 2.2728839610408613e-07, 8.776322601988795e-08, 6.826336118592735e-08, 3.5812259739032015e-05, 1.1646497100059605e-08, 7.173204608079686e-08, 1.182869496796357e-10, 8.809374207885412e-07, 1.5238903870340437e-05, 3.2645955798216164e-05, 1.4865344155623461e-06, 0.01966812089085579, 0.0020883092656731606, 2.556877689130488e-06, 0.00019726701430045068, 3.6002347769681364e-05, 7.294252282008529e-05, 8.671564799556108e-09, 3.4424303976265946e-06, 4.025610905955546e-06, 1.520348359917989e-05, 1.4297892647618937e-08, 7.17876855560462e-06, 8.982407706525919e-08, 3.2223366019934474e-07, 0.00021309306612238288, 3.1644215141568566e-06, 2.070694620215363e-07, 1.3433133062790148e-05], [1.0482123613542171e-08, 1.2119592440740234e-08, 2.910322427496226e-09, 2.457740606587322e-07, 9.001119849472161e-10, 9.995341443413963e-09, 0.0673917606472969, 2.4927367121563293e-06, 2.3627699796691104e-09, 1.7424882514660567e-07, 3.280210876255296e-06, 2.6445635015193147e-08, 9.343162687969198e-09, 9.257470900969622e-10, 3.2895336454785706e-10, 2.8284020459068415e-07, 2.2641288843772145e-09, 7.916522548612193e-08, 2.5291618044320785e-07, 2.8320473575149663e-06, 5.920895063127318e-08, 7.2178302445991e-08, 1.3794512199183373e-09, 2.493012196680411e-08, 1.2013970263069496e-07, 4.053156921202117e-09, 4.99163377298828e-07, 7.96762122945438e-09, 8.533413975442272e-09, 1.862403897767706e-09, 1.2235294721829604e-10, 5.902403188429162e-08, 4.242160503054038e-05, 1.5576578249465456e-08, 1.0289198826285428e-08, 3.0172006559858744e-10, 5.07635661506356e-07, 1.684906081322879e-08, 1.5739890502075582e-09, 1.0301500624754212e-10, 1.0163523711526068e-07, 1.1416410927722609e-08, 4.192156097815314e-07, 4.394200630031264e-07, 5.1042108317744805e-09, 2.9294278647284955e-07, 6.054970640434476e-07, 1.439552397641819e-05, 1.1170651958991584e-08, 1.9384969185409773e-09, 1.413270865668892e-06, 2.032631130077789e-08, 0.0005902072298340499, 5.635475419651925e-10, 8.669369042091368e-12, 2.7779049105447484e-07, 2.8567168630644346e-09, 3.53023665411456e-07, 1.9640390746644698e-05, 6.716247185067914e-07, 4.52852155774508e-08, 5.1155794877910665e-11, 2.2988584369443288e-09, 2.27695218235624e-09, 5.611998421528597e-09, 2.8442610755519127e-07, 2.2100021812576642e-09, 3.5263365560922466e-09, 3.333573417307889e-08, 1.0134269814443542e-06, 7.141680136157902e-09, 4.283471710664344e-08, 5.866493069106582e-08, 2.5010345357234787e-12, 7.802660206834844e-08, 7.87842452609766e-07, 3.167806639226001e-08, 1.5117768370487283e-08, 2.9757383224193745e-09, 3.068614459755814e-11, 4.316242030455442e-09, 4.464601843778837e-08, 2.875952986869379e-07, 2.074516714856145e-06, 1.1760924265719064e-12, 1.2731317111658313e-09, 3.000314663381687e-08, 4.074108872487159e-08, 1.4209240006834989e-08, 5.9175562228119816e-09, 2.630589222007984e-07, 1.3697525673705968e-06, 2.9018781333434163e-07, 7.227000248244408e-11, 1.1047518455598038e-09, 8.62418630731554e-07, 7.096798526617931e-08, 1.78401171524456e-06, 2.9948561859782785e-05, 1.2942263083459693e-06, 5.0760295522422894e-08, 1.7976204969016862e-09, 7.771051855343103e-07, 3.917723923052563e-09, 1.519393916238343e-10, 2.311009827948851e-09, 2.1347968868212774e-05, 1.102017677112599e-06, 8.081605074039544e-08, 0.0007324066828005016, 0.00035173678770661354, 6.871015330034425e-07, 0.0002702335477806628, 5.5960822642475705e-09, 1.0589412902106687e-08, 4.750592808022702e-09, 1.497268087291559e-08, 9.63510515816779e-09, 7.46193440370746e-10, 0.013934836722910404, 4.077389803569531e-08, 0.0011297636665403843, 1.2876968469299754e-07, 6.185937309055589e-06, 3.0444460207945667e-05, 7.3889854235176244e-09, 6.38999821944708e-08, 7.357617164416297e-07, 8.782741929280746e-07, 1.692689899357447e-08, 7.57554460051324e-07, 1.746058842400089e-06, 4.672865088650724e-06, 2.603191967542884e-11, 4.6391963337555353e-07, 4.843079715044496e-09, 3.764613953904927e-09, 3.172769720549695e-05, 6.800868277423433e-07, 3.0127672800972505e-08, 3.347460369695909e-05, 1.6378531597638357e-07, 1.1840147351449559e-07, 1.5132272324080986e-07, 5.59935017463431e-07, 9.163730396721803e-08, 3.8781786670938345e-09, 2.4358120853662513e-08, 9.692993216958712e-07, 3.6919158702408694e-11, 5.830367300063699e-10, 3.7478940839719144e-08, 1.3840494972328088e-08, 2.986319636022472e-09, 3.5539059695111064e-07, 5.698091776196179e-09, 2.716177789352514e-07, -0.16666652262210846, 5.23834877697027e-08, 6.904149785214031e-08, 1.2086673883970889e-08, 1.6742439656880492e-09, 6.039091005050068e-08, 7.538141653640196e-05, 7.144514529500157e-05, 4.292505309422268e-06, 6.055769574686565e-08, 1.8593868666982871e-09, 9.675464980318793e-07, 1.706996272332617e-06, 1.7739019142481993e-08, 0.08169332891702652, 1.6813121728187497e-10, 1.4416838389763598e-08, 7.678560223212116e-07, 2.8200380256748758e-05, 1.7356095227683e-08, 5.782325729342119e-08, 4.906404171833856e-08, 8.606944135181038e-08, 5.7925233704736456e-05, 4.487124556362687e-07, 3.586736019300929e-09, 1.5439827905083803e-07, 1.7464010682033404e-07, 3.085129027247113e-11, 8.679263037436602e-12, 4.726394138288015e-07, 5.6931468861876056e-05, 7.170462623662388e-08, 1.4168165307637537e-07, 2.150099875919409e-09, 6.962879717242743e-10, 1.2195645604506922e-09, 8.610769697270371e-08, 4.899925354351353e-09, 6.141343078525097e-07, 8.533962980727949e-10, 9.56051557365889e-11, 1.8047448258062815e-10], [1.182553721612578e-09, 2.7042838723900786e-07, 2.4469025916573628e-08, 2.057119452558709e-08, 4.888710421722953e-13, 3.9776278271119736e-08, 1.084993868971651e-06, 1.6268523950202507e-07, 3.8818802339246616e-11, 1.8751314200926572e-05, 4.160285698162625e-06, 1.8175556545929616e-11, 1.2106042959203478e-05, 9.673732925818967e-09, 1.675321215088843e-07, 5.536769931424601e-10, 1.4293619869931717e-06, 2.4654286789882462e-06, 4.370901649508596e-07, 3.8739221963624004e-07, 8.878991430982808e-10, 2.4942808067862643e-06, 1.2000329896455497e-10, 1.1810007549684087e-07, 3.139899263260304e-06, 4.764131062984234e-06, 5.605375719142103e-08, 6.719828525092453e-05, 2.119421878887806e-06, 2.651303248057957e-07, 7.042372240917416e-10, 7.190211022134463e-07, 3.413746014757635e-07, 1.2526533055279288e-07, 2.645368113007862e-05, 2.601919547873255e-11, 2.289865363991339e-08, 1.2340274579969446e-08, 1.1773099162226686e-09, 3.79138187511785e-09, 1.4259467207011767e-07, 8.012541002244689e-06, 7.755151614219358e-07, 4.322857805050262e-08, 4.9519851330614983e-08, 7.838261808501557e-06, 7.672640994771385e-11, 1.008506433919365e-08, 7.347344308072934e-06, 1.0157878449490454e-07, 1.0859618093661538e-08, 1.2852382269556983e-06, 7.349197517214634e-07, 1.1326810778200525e-07, 2.0127728816987656e-08, 9.862116678860389e-10, 2.1590162990747785e-09, 4.7337124442492495e-07, 1.5739613445475698e-05, 1.718329656341666e-07, 1.745614852666222e-08, 2.707404019020032e-05, 0.00028406124329194427, 4.701423108599556e-07, 1.5626578715455253e-06, 9.571677992425975e-07, 1.344436100225721e-06, 1.0003593786667153e-10, 1.617406439891056e-07, 6.106513961157134e-09, 1.3276981917442754e-05, 2.6566576707409695e-06, 7.146169537008973e-06, 1.5136903232584875e-12, 2.026091294737853e-07, 7.517370614884555e-10, 4.084206011611968e-06, 1.9609561197064806e-10, 1.4052665697761313e-08, 4.40548753033454e-08, 2.689143796530402e-09, 4.560471040804259e-07, 5.032642302893464e-09, 3.0654526028683904e-08, 4.6151857446830036e-08, 1.5710572398575096e-08, 0.00043729491881094873, 1.1602586710068863e-05, 2.725204062414832e-08, 2.3016342165504966e-09, 1.582242425968161e-08, 1.8361212994477683e-08, 1.951215722328925e-07, 1.8339479765927535e-06, 5.591638263524601e-10, 4.8530477414487905e-08, 8.601518608486458e-09, 2.3367063022305956e-06, 3.7877214253967395e-07, 0.0013270614435896277, 1.0492689384022924e-08, 3.1045956916386785e-07, 1.1089239393413663e-07, 0.0003915078123100102, 4.7631522193114506e-07, 8.651365419609647e-07, 2.0041209580767827e-08, 2.724167416090495e-06, 1.1179400738470235e-09, 9.318299731830848e-08, 1.7960962850338547e-07, 4.735995753435418e-05, 1.1550657319503443e-08, 1.1714470105061991e-07, 2.374650875935913e-07, 3.3348672445754346e-07, 0.008544021286070347, 2.7832149498863146e-05, 6.045060008119663e-09, 5.673264968919511e-09, 3.2876407658477547e-06, 3.1805929001649247e-09, 1.160338847228104e-08, 0.0002246328367618844, 1.456053269066615e-06, 2.324257184227463e-05, 0.007700094487518072, 4.8731848778515996e-08, 3.6183848806103924e-08, 1.0459200616708131e-08, 5.1129509301972575e-06, 5.8157287874394115e-09, 2.310832192264911e-09, 1.1715196279737938e-07, 4.7456497065390124e-11, 0.0076567367650568485, 2.0077894077985547e-05, 4.7292991744996016e-09, 3.64160854360307e-07, 5.678247368828826e-12, 7.640844756906517e-08, 2.2924589302419918e-06, 1.2092156111975783e-06, 1.3669323095655272e-07, 7.115156677173218e-07, 1.5930917696138636e-09, 7.876026220099419e-12, 5.725798928324366e-07, 5.2794324467697606e-11, 3.406005134820589e-06, 1.0964347652020479e-08, 1.788509621292178e-06, 3.149084548681458e-08, 7.830349346704679e-08, 2.3026476725362954e-08, 3.732851361348821e-09, 1.0416849249850202e-07, 2.7418725068173444e-08, 5.917812018196855e-09, 8.963979780673981e-08, 2.7110816702702323e-08, 2.5067396336453385e-07, 1.3893352956984018e-07, 1.309174422203796e-06, 1.7102851188610657e-06, 0.13804535567760468, 1.0517911164242832e-07, 1.4706968975986001e-08, 3.7223395565888495e-07, 2.2270889132869343e-08, 1.5885404991422547e-07, 1.3801637578580994e-05, 7.687414722568064e-07, 0.0009882606100291014, 1.4867768527437875e-07, 6.358765602954009e-08, 2.648271038196981e-05, 1.8905582876982407e-08, 1.5575390932554e-07, 1.6948617940215627e-06, 8.907048322726041e-05, 7.887526720651294e-08, 6.786885498044626e-10, 0.0003856367547996342, 3.6945237980035017e-09, 9.493952290995367e-10, 1.1533542192410096e-06, 5.378667111699542e-08, 6.212540260008836e-09, 2.5802428638144193e-08, 4.2037194702970737e-07, -0.1666140854358673, 6.314320216915803e-06, 4.358659862191416e-05, 2.3272625185200013e-05, 3.3934576038063824e-08, 1.3403956167223896e-08, 1.0372805547831376e-07, 7.75049002754713e-08, 4.634163815353531e-06]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[13.617902755737305, 13.607763290405273, -2.235592842102051, -7.135461330413818, -5.834252834320068, 8.101263999938965, -6.778660297393799, -7.489041805267334, -13.555184364318848, -10.957337379455566, 15.661747932434082, 7.9411468505859375, 8.90279769897461, -8.026673316955566, 0.20330914855003357, -1.5571342706680298, 3.6496450901031494, -10.655381202697754, 8.527759552001953, -2.9145543575286865, 1.3445806503295898, -1.4483709335327148, 6.5286946296691895, 10.487796783447266, -6.839400768280029, -3.2952961921691895, -5.590748310089111, 6.965519905090332, 0.24860912561416626, -5.3053669929504395, 1.3144818544387817, 1.9948883056640625, -8.608213424682617, -5.003154277801514, 9.677037239074707, -11.456511497497559, -19.853866577148438, -16.265913009643555, 0.7633937001228333, 4.2644758224487305, -6.917288303375244, -10.358043670654297, 2.551307201385498, 9.591402053833008, 2.1663663387298584, -9.729202270507812, 9.208040237426758, 19.033987045288086, 6.711559772491455, 1.38839852809906, 6.699204921722412, 6.103274345397949, -16.836809158325195, 2.424682855606079, -1.6148526668548584, -0.6483668088912964, -6.229921817779541, -7.217175483703613, 8.154349327087402, 0.545674741268158, 4.68479061126709, 8.818894386291504, 6.474362373352051, 3.183783531188965], [-1.1962569952011108, 0.8002221584320068, -7.604674816131592, 15.096689224243164, -5.899415969848633, -6.517509937286377, -9.390632629394531, -0.045072343200445175, -2.3547263145446777, 4.357944965362549, 7.129662990570068, -12.193704605102539, 2.73301100730896, 10.813203811645508, 18.20759391784668, -2.7810935974121094, -1.0335458517074585, 6.123360633850098, -15.932875633239746, -4.587059020996094, 4.845400810241699, 6.667115688323975, -8.781825065612793, -4.391801357269287, 1.2358731031417847, 10.76474666595459, 7.609373092651367, -0.5065773129463196, 16.631746292114258, 6.042945861816406, -2.500112771987915, 14.171996116638184, 5.902042388916016, 0.021870149299502373, -2.6501970291137695, -5.260589122772217, 5.097496032714844, 8.336068153381348, -12.507772445678711, 9.451851844787598, -8.033652305603027, -3.326770782470703, -10.927740097045898, -11.779650688171387, 1.6645485162734985, -22.866281509399414, 7.041996955871582, 17.565143585205078, 3.3790342807769775, -2.872880697250366, -0.6056869626045227, -3.7263097763061523, 22.76272964477539, -1.3603545427322388, 16.986711502075195, -6.9419426918029785, -4.526918888092041, 3.0896873474121094, -3.2885019779205322, -0.07025474309921265, 3.5973663330078125, 1.641909122467041, -3.5721869468688965, -5.926572322845459], [13.890809059143066, -6.692887306213379, 9.428009033203125, 6.147580146789551, -13.777595520019531, 2.423327922821045, 1.3516188859939575, 8.26119613647461, 7.155887603759766, 7.6204328536987305, -1.0105098485946655, 7.714442729949951, -28.49640655517578, -1.8641276359558105, -3.6890666484832764, -2.280222177505493, 3.4728915691375732, -13.047534942626953, 6.546293258666992, 10.448461532592773, -3.4222817420959473, 7.943665981292725, 7.85800313949585, -4.55166482925415, -9.76500415802002, -4.061334609985352, 12.18052864074707, 6.463390350341797, 3.522480010986328, 0.5380439162254333, -4.860867500305176, -8.43025016784668, -0.807099461555481, 3.163252115249634, 2.102304220199585, -10.340996742248535, -14.914159774780273, -19.24861717224121, -0.7370598316192627, 2.1281189918518066, -11.438316345214844, -6.0189595222473145, -9.99161148071289, 16.30776023864746, -12.186837196350098, 12.094727516174316, 6.330288410186768, 3.7070298194885254, -0.04826715216040611, 4.934989929199219, 13.564637184143066, 9.057352066040039, -4.26738977432251, 1.2421718835830688, 5.76271390914917, -4.361003875732422, -19.13851547241211, 10.899384498596191, 12.17440414428711, -18.564306259155273, 13.075104713439941, -15.845211029052734, -1.472341537475586, -19.875465393066406], [9.00789737701416, 4.609238147735596, -10.39737606048584, -6.389075756072998, 0.5880317091941833, 2.5275027751922607, -14.17315673828125, 14.879905700683594, 7.502213478088379, -11.068974494934082, -2.736877918243408, -5.46295166015625, -2.120407819747925, 4.889978408813477, 9.784646034240723, -18.345993041992188, -2.120786666870117, -21.80599594116211, 7.078749179840088, -2.2543258666992188, 0.6349616050720215, -8.41419506072998, 4.3215861320495605, -9.766908645629883, -1.8878533840179443, -6.7544097900390625, -0.9650703072547913, 7.214974880218506, 5.117284774780273, -8.530686378479004, -2.646007776260376, -6.9703049659729, -10.184823036193848, 14.121015548706055, -3.9383184909820557, -16.236345291137695, 10.978293418884277, 23.50222396850586, 10.37680721282959, 9.784217834472656, -5.635745048522949, 14.274239540100098, 11.198040962219238, -8.379125595092773, -1.706150770187378, -0.2894168794155121, -34.957096099853516, -1.276336908340454, -15.903410911560059, -9.33160400390625, 17.4683780670166, -0.37474483251571655, -16.004352569580078, 15.430888175964355, -12.461166381835938, 1.1528269052505493, 1.3778045177459717, -12.56631088256836, -8.343598365783691, 10.18616771697998, -18.605833053588867, -6.083755016326904, -17.4868221282959, 5.839540481567383], [-7.029821395874023, -11.264765739440918, -6.420123100280762, 15.378868103027344, 4.53043794631958, 15.789521217346191, -3.817988395690918, -6.429345607757568, -0.042731642723083496, -4.991672515869141, 9.981595993041992, -4.971255302429199, -1.9572272300720215, 8.71102237701416, 11.57840347290039, -1.7013227939605713, -17.061471939086914, -6.338637828826904, -10.499300003051758, -13.323578834533691, -7.66091251373291, 0.773909866809845, 3.070188522338867, 3.397017002105713, 3.2779362201690674, -14.5954008102417, 10.504385948181152, -4.457089900970459, 0.9658531546592712, -1.4594520330429077, 9.601432800292969, -0.2587074041366577, -6.780991077423096, 1.7141152620315552, 6.246094703674316, 8.826475143432617, 6.88646125793457, -15.722633361816406, -1.2942003011703491, -6.827544212341309, 7.7717390060424805, -0.4716409146785736, 16.34638023376465, -1.8502166271209717, 4.708487510681152, -3.963813066482544, 8.307933807373047, 0.9154368042945862, 3.6571595668792725, -5.452734470367432, -18.299421310424805, 5.551312446594238, -9.537625312805176, -6.896496772766113, 22.50204086303711, 12.662325859069824, -6.928596019744873, -4.452495098114014, 25.84244155883789, -2.123065948486328, 6.392748832702637, -4.047876358032227, 11.895559310913086, 3.7206039428710938], [0.6726235151290894, -7.256855487823486, -4.139512538909912, -11.174577713012695, 4.576958179473877, -19.125030517578125, 4.150967121124268, -8.868558883666992, 6.560260772705078, -13.604686737060547, -8.001035690307617, 18.43229103088379, -11.658834457397461, -5.419623374938965, 4.148810863494873, -12.44397258758545, 8.81540584564209, 5.669585227966309, -2.251213788986206, 4.530974388122559, -14.803897857666016, -9.721516609191895, 8.829864501953125, -13.529358863830566, -11.182917594909668, 0.765573263168335, -3.7332303524017334, 6.18486213684082, -23.613384246826172, 2.7998204231262207, -9.24383544921875, 2.3265926837921143, -10.592802047729492, -12.382628440856934, -7.000410079956055, 6.327972412109375, -3.6208715438842773, 7.319422721862793, -3.9191324710845947, -7.780095100402832, -25.291030883789062, -5.12026834487915, 5.489223480224609, 8.174155235290527, -5.460629940032959, 16.872190475463867, 8.159024238586426, -5.593743324279785, -6.103876113891602, -20.545166015625, 6.096941947937012, 3.1032772064208984, 12.669273376464844, 0.7347588539123535, 7.503450870513916, -12.52999210357666, 3.9996135234832764, 5.895607948303223, 12.83988094329834, -4.805196285247803, -16.244035720825195, 14.207104682922363, 8.777968406677246, -2.1612908840179443], [4.8652753829956055, 16.009891510009766, 1.9991023540496826, -7.662238597869873, -15.212532997131348, -5.416059494018555, -17.166603088378906, 1.4608439207077026, 7.740931034088135, 4.542599201202393, -1.3420909643173218, -5.435003280639648, 9.739378929138184, -4.6323018074035645, -3.7955222129821777, 9.977829933166504, -11.663362503051758, -14.408849716186523, 7.231762409210205, -22.135255813598633, -3.8707149028778076, -3.417605400085449, -7.237608909606934, 20.3496036529541, -2.752319574356079, -12.629704475402832, -2.339721918106079, -1.5631251335144043, -14.395133972167969, 15.870065689086914, -6.4965057373046875, 4.153934478759766, 5.893619060516357, 0.20251812040805817, 2.7144484519958496, -0.3951033651828766, 22.967737197875977, -8.39831829071045, 7.2539801597595215, -3.4345028400421143, 5.144965171813965, -15.845132827758789, -7.692773342132568, 0.6496701836585999, 0.1283462941646576, 0.12703090906143188, -28.834863662719727, 1.3466078042984009, -15.344507217407227, 8.49579906463623, -0.4424910545349121, -10.498307228088379, -9.444561958312988, 6.509977340698242, -0.4285990297794342, 4.697898864746094, -0.5570217967033386, 3.7346279621124268, 13.368712425231934, -3.285853385925293, -0.7561139464378357, 14.48094367980957, -0.7699221968650818, -13.947486877441406], [10.245674133300781, -2.437509059906006, -13.445899963378906, 3.7780067920684814, 0.652916669845581, -20.072147369384766, 10.311320304870605, -9.265921592712402, -7.457694053649902, 12.81632137298584, -1.640013337135315, 1.174573302268982, 5.079190254211426, 4.9687042236328125, -3.747758388519287, -12.818099975585938, 2.649017810821533, -8.083002090454102, 13.799576759338379, 9.239899635314941, -2.003390073776245, -14.350926399230957, 1.1041100025177002, 16.771440505981445, -8.411264419555664, -3.515507936477661, 5.395540714263916, 19.258281707763672, 5.322253227233887, 2.7704567909240723, -5.759885787963867, 9.439209938049316, -1.7516309022903442, 12.338088035583496, -6.2302021980285645, -3.2319869995117188, 14.740653038024902, 2.3771817684173584, 8.296748161315918, -16.639842987060547, -2.7736990451812744, -9.538782119750977, -4.906532287597656, 4.915359020233154, -5.106744289398193, -15.940852165222168, 3.8796353340148926, -4.629941940307617, -4.551705837249756, -14.069239616394043, -3.7770836353302, -1.840502142906189, -13.034257888793945, 8.000103950500488, -10.178040504455566, 0.5580968260765076, -1.6815474033355713, 1.0975196361541748, 3.4289090633392334, 1.2931225299835205, -6.200167655944824, -8.06570816040039, -7.667277812957764, -0.9719274044036865]], "dtype": "float32"}, {"__nd__": [-100, -100, -100, 36, 11, 44, 34, 13], "dtype": "int64"}], "e": {"__tup__": [20.4377721786499, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0113134862876905e-07, 1.243289138308512e-09, 3.7781761675468217e-16, 2.0800084948153184e-14, 2.2293826507091552e-11, 1.5505502437562768e-10, 8.659282339868492e-18, 3.5897697671316564e-05, 2.2437536273400838e-08, 1.930241047166863e-16, 8.020379065748484e-13, 5.2514094553987936e-14, 1.4856774664673278e-12, 1.6462519125681752e-09, 2.1989910692354897e-07, 1.3342650755148788e-19, 1.4851136813376353e-12, 4.193551250670262e-21, 1.4691535099586872e-08, 1.2994643338801182e-12, 2.3365019988230706e-11, 2.7451541609541455e-15, 9.324958583079024e-10, 7.097253066143524e-16, 1.8746545114856072e-12, 1.4434534093715236e-14, 4.7171619908803475e-12, 1.683564043730712e-08, 2.066399051869894e-09, 2.4432957589917096e-15, 8.783329966122189e-13, 1.1631638543779712e-14, 4.672976479070344e-16, 1.6806790881673805e-05, 2.4122134953570573e-13, 1.1001443210336624e-18, -0.19999927282333374, 0.19938470423221588, 3.975536628786358e-07, 2.1980494580020604e-07, 4.418066593451196e-14, 1.9589760995586403e-05, 9.03759882930899e-07, 2.8431359676772076e-15, 2.2481957701742106e-12, 9.270717006570894e-12, 8.149514011404279e-27, 3.4554106537881824e-12, 1.5347641434206884e-18, 1.096835563014684e-15, 0.0004777776775881648, 8.512482385647502e-12, 1.3874061565950059e-18, 6.22809529886581e-05, 4.797222381972708e-17, 3.9216765251870456e-11, 4.9110937538898725e-11, 4.318433038070275e-17, 2.9459634381332747e-15, 3.285504703853803e-07, 1.028953745397322e-19, 2.822696808550077e-14, 3.150475819694298e-19, 4.254865793029694e-09], [1.0223686614568221e-15, 1.480448651986374e-17, 1.881026394758359e-15, 5.51539824300562e-06, 1.0719223425548208e-10, 8.316130333696492e-06, 2.5379766783193217e-14, 1.863762804502132e-15, 1.1067791648444736e-12, 7.84808212883105e-15, 2.4978790591489997e-08, -0.20000000298023224, 1.6315716681349068e-13, 7.010804381479829e-09, 1.2332640153545071e-07, 2.107387208532055e-13, 4.496922486822213e-20, 2.0407213279524748e-15, 3.182954601157098e-17, 1.889134935283306e-18, 5.439108780136794e-16, 2.5045193783462816e-12, 2.488772903042058e-11, 3.450840654894982e-11, 3.063439085737585e-11, 5.295621138310023e-19, 4.213239890304976e-08, 1.3394594474921452e-14, 3.0344771834167616e-12, 2.684031111515467e-13, 1.7079242908835113e-08, 8.917934750038436e-13, 1.311210919877038e-15, 6.412835106689441e-12, 5.960235083257714e-10, 7.868805163013803e-09, 1.1307627056922342e-09, 1.7154040523851717e-19, 3.16632164281197e-13, 1.2515672007752448e-15, 2.7405839819749644e-09, 7.207555949642619e-13, 1.4513162568619009e-05, 1.815853511392268e-13, 1.2808253069263031e-10, 2.1935973071080146e-14, 4.68499505856812e-09, 2.885285977152763e-12, 4.476139944808999e-11, 4.9491093257543646e-15, 1.3040146119935458e-20, 2.9752486585543636e-10, 8.326868561057727e-17, 1.1681786070630719e-15, 0.006841187831014395, 3.6458399677030684e-07, 1.1312747579797413e-15, 1.3456281343371141e-14, 0.19312971830368042, 1.382240269508575e-13, 6.901687554794478e-10, 2.0167356418751906e-14, 1.693538536073902e-07, 4.76932694726262e-11], [3.144229321350167e-09, 1.1318420159445175e-12, 2.5564082101192653e-11, 2.2508184484531894e-14, 1.5600745939536864e-07, 7.934206420587172e-18, 1.0189192778398137e-07, 2.2585603634724116e-13, 1.133660248342494e-06, 1.9813294952733713e-15, 5.377646570235706e-13, 0.16234546899795532, 1.3868529737700967e-14, 7.106991210814595e-12, 1.0167256192517016e-07, 6.32484160395996e-15, 1.0811351785378065e-05, 4.65229732071748e-07, 1.6893066112633193e-10, 1.4899615052854642e-07, 5.972358901476331e-16, 9.624905175165469e-14, 1.0968805327138398e-05, 2.136346053741565e-15, 2.2321269724061288e-14, 3.4505016621722007e-09, 3.837747827861726e-11, 7.788420361976023e-07, 8.917324950869588e-20, 2.6384219253827723e-08, 1.5518550257765523e-13, 1.6437075700537207e-08, 4.0271927367321533e-14, 6.724981662308786e-15, 1.4627105950071329e-12, 8.986723400994379e-07, 4.294116104364143e-11, 2.4220482828241074e-06, 3.186694311207994e-11, 6.707258988665477e-13, 1.6658793920184295e-20, 9.587245060338478e-12, 3.884519799157715e-07, 5.693619186786236e-06, -0.20000000298023224, 0.03411120921373367, 5.608117135125212e-06, 5.9712547406665095e-12, 3.5852395271346316e-12, 1.9175490029148087e-18, 7.132900350370619e-07, 3.573831364178659e-08, 0.0005100272828713059, 3.3457954184967775e-09, 2.9114207791280933e-06, 5.803514000748619e-15, 8.758063074765232e-08, 5.832139322592411e-07, 0.0006049052462913096, 1.313794472518337e-11, 1.4148252045317655e-16, 0.002373914234340191, 1.0414085409138352e-05, 1.8482515518059017e-10], [2.5575450646186937e-09, 0.00017695751739665866, 1.4556610083982235e-10, 9.272424780846621e-15, 4.87687822060034e-18, 8.763886019886105e-14, 6.910370234844562e-19, 8.49763245880375e-11, 4.536332554039291e-08, 1.8521962852346974e-09, 5.152273573061361e-12, 8.599427458099926e-14, 3.346724781749799e-07, 1.91901589020535e-13, 4.4308483206252636e-13, 4.247935692092142e-07, 1.696396955649224e-16, 1.089375360036411e-17, 2.7263133972610376e-08, 4.804431403182776e-21, 4.109903606677734e-13, 6.465687360512895e-13, 1.4177781466743512e-14, 0.013570068404078484, 1.2576053479845806e-12, 6.454313633972434e-17, 1.899911001093657e-12, 4.130519277306677e-12, 1.1044220235329344e-17, 0.0001538662618258968, 2.974848828645599e-14, 1.255718418136098e-09, 7.152006986643755e-09, 2.4144271651427296e-11, -0.20000000298023224, 1.328220519680734e-11, 0.18604713678359985, 4.441386765741823e-15, 2.7875664443399728e-08, 6.35734141111316e-13, 3.382914615102095e-09, 2.5906438585275303e-18, 8.993572217248086e-15, 3.775803628092156e-11, 2.241822352755385e-11, 2.238878180071957e-11, 5.916151499987268e-24, 7.58028570468916e-11, 4.27392269746298e-18, 9.650283772089097e-08, 1.2667469503901962e-11, 5.438813377396439e-16, 1.5600549060993665e-15, 1.324673082336858e-08, 1.2844687119484277e-11, 2.16338102987379e-09, 1.1296653716630356e-11, 8.256388350780242e-10, 1.2613031685759779e-05, 7.376220944807499e-13, 9.257329278145043e-12, 3.835794996120967e-05, 9.130371805832205e-12, 1.7280077620196116e-17], [2.2130616343929432e-05, 6.866838347940885e-11, 1.1372963152017721e-15, 3.436538165146885e-08, 1.5097726402402145e-09, 1.5070613477184047e-18, 2.3632150259800255e-05, 7.433866076964751e-14, 4.534380683429706e-13, 0.00028934195870533586, 1.5244222550947484e-10, 2.5436932560296555e-09, 1.262461353235267e-07, -0.1999998837709427, 1.852348829878281e-11, 2.1307067201369826e-15, 1.1112387454659256e-08, 2.4263338734009654e-13, 0.0007734528044238687, 8.094521035673097e-06, 1.0599681549150475e-10, 4.600722451087184e-16, 2.370625473702148e-09, 0.01510420348495245, 1.7473827560680194e-13, 2.336618225295961e-11, 1.7322362566574157e-07, 0.181601420044899, 1.609826085768873e-07, 1.2547223704473254e-08, 2.4766642729712185e-12, 9.879854587779846e-06, 1.3634215978441944e-10, 0.00017935657524503767, 1.5474317790684089e-12, 3.102556059286776e-11, 0.0019821571186184883, 8.467414325252776e-09, 3.1519964522885857e-06, 4.664034722823095e-17, 4.906272610405438e-11, 5.658656458988262e-14, 5.813984276653583e-12, 1.071685318265736e-07, 4.759076379506899e-12, 9.382730202268184e-17, 3.80415130507572e-08, 7.666471624201421e-12, 8.290349309825196e-12, 6.097618133716297e-16, 1.7988166511884174e-11, 1.2474811461604673e-10, 1.7165112088196556e-15, 2.34290541811788e-06, 2.985983246143846e-14, 1.3731934478400376e-09, 1.462403115271016e-10, 2.3550503769342868e-09, 2.4238746121341137e-08, 2.8638487137300217e-09, 1.5946109450243329e-12, 2.468659499911541e-13, 3.6770293840998614e-13, 2.9733834838729933e-10]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(cross_entropy_fwd_bwd, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")